In [1]:
import os,warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score,KFold
from xgboost import XGBRegressor
warnings.filterwarnings("ignore")

meta_dir=r"C:\Users\Arpit Joshua Elias\OneDrive\Desktop\Mentor submissions\Pharma datasets"
proc=pd.read_csv(os.path.join(meta_dir,"Process.csv"),sep=";")
lab=pd.read_csv(os.path.join(meta_dir,"Laboratory.csv"),sep=";")
targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]
print("Process.csv:",proc.shape,"| Laboratory.csv:",lab.shape)

Process.csv: (1005, 35) | Laboratory.csv: (1005, 55)


In [2]:
df=proc.merge(lab[["batch"]+targets],on="batch",how="inner")
exclude=["batch","code","Drug release average (%)","Drug release min (%)","Residual solvent","Total impurities","Impurity O","Impurity L"]+targets
feat_cols=[c for c in df.columns if c not in exclude]
X=df[feat_cols].apply(pd.to_numeric,errors="coerce").fillna(0).values
print(f"{len(feat_cols)} engineered features used (product identity excluded):")
print(feat_cols)

27 engineered features used (product identity excluded):
['tbl_speed_mean', 'tbl_speed_change', 'tbl_speed_0_duration', 'total_waste', 'startup_waste', 'weekend', 'fom_mean', 'fom_change', 'SREL_startup_mean', 'SREL_production_mean', 'SREL_production_max', 'main_CompForce mean', 'main_CompForce_sd', 'main_CompForce_median', 'pre_CompForce_mean', 'tbl_fill_mean', 'tbl_fill_sd', 'cyl_height_mean', 'stiffness_mean', 'stiffness_max', 'stiffness_min', 'ejection_mean', 'ejection_max', 'ejection_min', 'Startup_tbl_fill_maxDifference', 'Startup_main_CompForce_mean', 'Startup_tbl_fill_mean']


In [3]:
rows=[]
for t in targets:
    y=pd.to_numeric(df[t],errors="coerce").fillna(df[t].median()).values
    Xs=StandardScaler().fit_transform(X)
    kf=KFold(5,shuffle=True,random_state=42)
    lasso=-cross_val_score(Lasso(alpha=0.1),Xs,y,cv=kf,scoring="neg_root_mean_squared_error").mean()
    xgb=-cross_val_score(XGBRegressor(n_estimators=100,max_depth=4,verbosity=0),X,y,cv=kf,scoring="neg_root_mean_squared_error").mean()
    mean_rmse=np.sqrt(((y-y.mean())**2).mean())
    headroom="small" if xgb>0.85*mean_rmse else ("large" if xgb<0.5*mean_rmse else "moderate")
    rows.append([t,round(lasso,3),round(xgb,3),round(mean_rmse,3),headroom])
res=pd.DataFrame(rows,columns=["target","LASSO_RMSE","XGB_RMSE","predict_mean_RMSE","RQ1_headroom"])
print(res.to_string(index=False))

         target  LASSO_RMSE  XGB_RMSE  predict_mean_RMSE RQ1_headroom
 dissolution_av       2.922     2.854              3.364     moderate
tbl_av_hardness       5.447     3.826             12.671        large
 tbl_rsd_weight       0.519     0.564              0.571        small
    fct_tensile       0.297     0.156              0.373        large


In [4]:
corr=df[targets].apply(pd.to_numeric,errors="coerce").corr()
print("Target correlation matrix:")
print(corr.round(2).to_string())
print()
print("Coupled targets (|correlation| >= 0.3), candidates for multi-task benefit:")
for i in range(len(targets)):
    for j in range(i+1,len(targets)):
        c=corr.iloc[i,j]
        if abs(c)>=0.3:
            print(f"  {targets[i]} and {targets[j]}: {c:.2f}")

Target correlation matrix:
                 dissolution_av  tbl_av_hardness  tbl_rsd_weight  fct_tensile
dissolution_av             1.00            -0.28            0.02         0.52
tbl_av_hardness           -0.28             1.00           -0.06        -0.38
tbl_rsd_weight             0.02            -0.06            1.00        -0.02
fct_tensile                0.52            -0.38           -0.02         1.00

Coupled targets (|correlation| >= 0.3), candidates for multi-task benefit:
  dissolution_av and fct_tensile: 0.52
  tbl_av_hardness and fct_tensile: -0.38


In [5]:
code_counts=lab.groupby("code")["batch"].nunique().sort_values(ascending=False)
eligible=code_counts[code_counts>=50]
print("Batches per product code (top 10):")
print(code_counts.head(10).to_string())
print()
print("Codes with >= 50 batches (OOD-eligible):",len(eligible))
print("Codes:",list(eligible.index))

Batches per product code (top 10):
code
17    207
23    187
13    131
1      95
21     68
15     64
22     41
25     34
10     27
4      22

Codes with >= 50 batches (OOD-eligible): 6
Codes: [17, 23, 13, 1, 21, 15]
